## Exploratory Data Analysis

- **Objective**: 
    - check data quality
    - check row and columns count/shape
    - understand the data

### Data dictionary

| Column Name                | Description                                                              | Use                                                                                  |
|----------------------------|--------------------------------------------------------------------------|-------------------------------------------------------------------------------------|
| trans_date_trans_time      | The date and time when the transaction occurred.                        | Used for time-series analysis and understanding transaction patterns over time.     |
| cc_num                     | The credit card number of the customer (masked or anonymized for privacy). | Identifies which customer made the transaction, useful for tracking customer behavior. |
| merchant                   | The name of the merchant where the transaction took place.              | Helps analyze spending habits by merchant and identify potentially fraudulent merchants. |
| category                   | The category of the merchant (e.g., miscellaneous, grocery, etc.).     | Used for categorizing transactions and analyzing spending by category.               |
| amt                        | The amount of money involved in the transaction.                        | Important for calculating total spending, fraud detection, and transaction analysis. |
| first                      | The first name of the credit card holder.                               | Useful for personalization and customer segmentation.                               |
| last                       | The last name of the credit card holder.                                | Similar to the first name, helps in personalization and customer identification.    |
| gender                     | The gender of the credit card holder.                                   | Used for demographic analysis and understanding spending patterns by gender.        |
| street                     | The street address of the credit card holder.                           | Provides location data for geographic analysis of transactions.                     |
| city                       | The city of the credit card holder.                                     | Used for geographic analysis and understanding local spending trends.              |
| state                      | The state of the credit card holder.                                    | Important for regional analysis and fraud detection.                                |
| zip                        | The zip code of the credit card holder.                                 | Used for local analysis and understanding demographic distributions.                 |
| lat                        | The latitude of the credit card holder's location.                      | Used for mapping transactions and analyzing geographic trends.                      |
| long                       | The longitude of the credit card holder's location.                     | Similar to latitude, helps in geographic analysis.                                  |
| city_pop                   | The population of the city where the credit card holder resides.        | Useful for demographic analysis and understanding market size.                      |
| job                        | The job title of the credit card holder.                                | Provides insights into spending patterns based on occupation.                       |
| dob                        | The date of birth of the credit card holder.                            | Used for age analysis and understanding customer demographics.                      |
| trans_num                  | A unique identifier for the transaction.                                | Essential for tracking individual transactions and preventing duplicates.            |
| unix_time                  | The UNIX timestamp of the transaction.                                  | Useful for time-based operations and calculations.                                  |
| merch_lat                  | The latitude of the merchant's location.                                | Important for analyzing merchant locations and trends.                              |
| merch_long                 | The longitude of the merchant's location.                               | Similar to merch_lat, helps in geographic analysis.                                 |
| is_fraud                   | A flag indicating whether the transaction is fraudulent (1 for fraud, 0 for legitimate). | Essential for training fraud detection models and evaluating transaction legitimacy.  |



In [52]:
import numpy as np
import pandas as pd
import os
import json

# Get the current working directory
current_path = os.getcwd()
print("Current Working Directory:", current_path)

Current Working Directory: /home/achum/de_course/goal_1/portfolio/Project1_Document_Streaming/code/client


In [53]:
# read data
data_loc = '../../data/fraudTrain.csv'

df = pd.read_csv(data_loc,index_col=0).head(5)
df.head(1).T

,0
trans_date_trans_time,2019-01-01 00:00:18
cc_num,2703186189652095
merchant,"fraud_Rippin, Kub and Mann"
category,misc_net
amt,4.97
first,Jennifer
last,Banks
gender,F
street,561 Perry Cove
city,Moravian Falls


In [54]:
s = df.isnull().sum() > 0
print(f'No of columns with missing rows: {len(s[s])}')
print(f"Check df shape -> rows,columns: {df.shape}")

No of columns with missing rows: 0
Check df shape -> rows,columns: (5, 22)


In [55]:
# check datatype
df.dtypes

trans_date_trans_time     object
cc_num                     int64
merchant                  object
category                  object
amt                      float64
first                     object
last                      object
gender                    object
street                    object
city                      object
state                     object
zip                        int64
lat                      float64
long                     float64
city_pop                   int64
job                       object
dob                       object
trans_num                 object
unix_time                  int64
merch_lat                float64
merch_long               float64
is_fraud                   int64
dtype: object

In [56]:
# convert to json
df2 = df.head(5).copy()
# df2['json'] = df2.to_json(orient='records', lines=True).splitlines()

In [57]:
# Transforming to desired JSON structure
features = []
for _, row in df2.iterrows():
    transformed = {
        "trans_date_trans_time": row['trans_date_trans_time'],
        "cc_num": row['cc_num'],
        "merchant": row['merchant'],
        "category": row['category'],
        "amt": row['amt'],
        "card_holder": {
            "first": row['first'],
            "last": row['last'],
            "gender": row['gender'],
            "dob": row['dob']
        },
        "address": {
            "street": row['street'],
            "city": row['city'],
            "state": row['state'],
            "zip": row['zip'],
            "lat": row['lat'],
            "long": row['long']
        },
        "city_pop": row['city_pop'],
        "job": row['job'],
        "trans_num": row['trans_num'],
        "unix_time": row['unix_time'],
        "merch_location": {
            "lat": row['merch_lat'],
            "long": row['merch_long']
        },
        "is_fraud": row['is_fraud']
    }
    features.append(transformed)

# Convert to JSON string
# df2['json'] = [json.dumps(feature, indent=4) for feature in features]
df2['json'] = [json.dumps(feature, separators=(',', ':')) for feature in features]
# Beautify the compact JSON
df2['beautified_json'] = df2['json'].apply(lambda x: json.dumps(json.loads(x), indent=4))



In [58]:
df2['beautified_json'].head()

0    {\n    "trans_date_trans_time": "2019-01-01 00...
1    {\n    "trans_date_trans_time": "2019-01-01 00...
2    {\n    "trans_date_trans_time": "2019-01-01 00...
3    {\n    "trans_date_trans_time": "2019-01-01 00...
4    {\n    "trans_date_trans_time": "2019-01-01 00...
Name: beautified_json, dtype: object

In [59]:
df2.to_csv('transactions.csv', index=False)